In [4]:
import cv2
import numpy as np
import mediapipe as mp
import joblib
from collections import deque

In [5]:
model = joblib.load('banminton_pose_model_ann.pkl')
scale = joblib.load('scaler.pkl')


In [6]:
mp_pose = mp.solutions.pose
mp_draw = mp.solutions.drawing_utils
pose = mp_pose.Pose()

In [7]:
class_name = ['Backhand', 'Forehand_Defense', 'Smash', 'Stance']

In [8]:
history = deque(maxlen=10)

In [ ]:
def dataset(lmlist):


    nose = lmlist[0][1:]

    left_shoulder = lmlist[11][1:]
    right_shoulder = lmlist[12][1:]

    left_elbow = lmlist[13][1:]
    right_elbow = lmlist[14][1:]

    left_wrist = lmlist[15][1:]
    right_wrist = lmlist[16][1:]

    left_hip = lmlist[23][1:]
    right_hip = lmlist[24][1:]

    left_knee = lmlist[25][1:]
    right_knee = lmlist[26][1:]

    left_ankle = lmlist[27][1:]
    right_ankle = lmlist[28][1:]


    mid_hip_x = (left_hip[0] + right_hip[0]) / 2


    left_hand_up = 1 if left_wrist[1] < left_shoulder[1] else 0
    right_hand_up = 1 if right_wrist[1] < right_shoulder[1] else 0

  
    left_hand_above_head = 1 if left_wrist[1] < nose[1] else 0
    right_hand_above_head = 1 if right_wrist[1] < nose[1] else 0


    left_arm_stretched = 1 if (
        abs(left_wrist[0] - left_shoulder[0]) >
        abs(left_elbow[0] - left_shoulder[0])
    ) else 0

    right_arm_stretched = 1 if (
        abs(right_wrist[0] - right_shoulder[0]) >
        abs(right_elbow[0] - right_shoulder[0])
    ) else 0


    left_arm_bent = 1 if (
        abs(left_elbow[0] - left_shoulder[0]) >
        abs(left_wrist[0] - left_shoulder[0])
    ) else 0

    right_arm_bent = 1 if (
        abs(right_elbow[0] - right_shoulder[0]) >
        abs(right_wrist[0] - right_shoulder[0])
    ) else 0


    left_hand_inside = 1 if left_wrist[0] > left_elbow[0] else 0
    right_hand_inside = 1 if right_wrist[0] < right_elbow[0] else 0


    left_cross_body = 1 if left_wrist[0] > nose[0] else 0
    right_cross_body = 1 if right_wrist[0] < nose[0] else 0


    left_elbow_up = 1 if left_elbow[1] < left_shoulder[1] else 0
    right_elbow_up = 1 if right_elbow[1] < right_shoulder[1] else 0

    lean_left = 1 if nose[0] < mid_hip_x else 0
    lean_right = 1 if nose[0] > mid_hip_x else 0


    legs_open = 1 if (
        left_ankle[0] < left_knee[0] and
        right_ankle[0] > right_knee[0]
    ) else 0


    left_knee_bent = 1 if (
        abs(left_knee[0] - left_hip[0]) >
        abs(left_ankle[0] - left_knee[0])
    ) else 0

    right_knee_bent = 1 if (
        abs(right_knee[0] - right_hip[0]) >
        abs(right_ankle[0] - right_knee[0])
    ) else 0


    sitting = 1 if (
        left_hip[1] > left_knee[1] or
        right_hip[1] > right_knee[1]
    ) else 0


    shoulders_level = 1 if abs(
        left_shoulder[1] - right_shoulder[1]
    ) < 0.05 else 0


    hips_level = 1 if abs(
        left_hip[1] - right_hip[1]
    ) < 0.05 else 0


    left_wrist_outside = 1 if left_wrist[0] < left_shoulder[0] else 0
    right_wrist_outside = 1 if right_wrist[0] > right_shoulder[0] else 0


    left_hand_below_hip = 1 if left_wrist[1] > left_hip[1] else 0
    right_hand_below_hip = 1 if right_wrist[1] > right_hip[1] else 0


    row = [

        left_hand_up,
        right_hand_up,

        left_hand_above_head,
        right_hand_above_head,

        left_arm_stretched,
        right_arm_stretched,

        left_arm_bent,
        right_arm_bent,

        left_hand_inside,
        right_hand_inside,

        left_cross_body,
        right_cross_body,

        left_elbow_up,
        right_elbow_up,

        lean_left,
        lean_right,

        shoulders_level,
        hips_level,

        legs_open,

        left_knee_bent,
        right_knee_bent,

        sitting,

        left_wrist_outside,
        right_wrist_outside,

        left_hand_below_hip,
        right_hand_below_hip
    ]

    return row



In [ ]:
video = cv2.VideoCapture(0)

video.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
video.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

required_landmarks = [
    0,   # nose
    11, 12,   # shoulders
    13, 14,   # elbows
    15, 16,   # wrists
    23, 24,   # hips
    25, 26,   # knees
    27, 28    # ankles
]

while True:

    ret, img = video.read()

    if not ret:
        break

    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    result = pose.process(rgb)

    landmark_visible = True

    if result.pose_landmarks:

        mp_draw.draw_landmarks(
            img,
            result.pose_landmarks,
            mp_pose.POSE_CONNECTIONS
        )

        lmlist = []

        for idx, lm in enumerate(result.pose_landmarks.landmark):
            lmlist.append([idx, lm.x, lm.y, lm.z, lm.visibility])

        for idx in required_landmarks:

            visibility = lmlist[idx][4]

            # visibility threshold
            if visibility < 0.60:
                landmark_visible = False
                break


        if not landmark_visible:

            cv2.putText(
                img,
                "Landmark Not Visible",
                (20, 50),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (0, 0, 255),
                3
            )

        else:

            try:

                row = dataset(lmlist)


                row = np.array(row).reshape(1, -1)

                row = scale.transform(row)


                prediction = model.predict(row)[0]

                pred_class = np.argmax(prediction)

                history.append(pred_class)

                final_pred = max(set(history), key=history.count)

                max_prob = np.max(prediction)


                label = (
                    class_name[final_pred]
                    if max_prob >= 0.70
                    else "Uncertain"
                )


                cv2.putText(
                    img,
                    label,
                    (10, 40),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1,
                    (0, 255, 0),
                    2
                )


                bar_x = 170
                bar_width_max = 200
                start_y = 80

                for i, val in enumerate(prediction):

                    y = start_y + i * 35

                    percent = int(val * 100)

                    bar_width = int(val * bar_width_max)

                    cv2.putText(
                        img,
                        class_name[i],
                        (10, y),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.6,
                        (255, 255, 255),
                        1
                    )

                    cv2.rectangle(
                        img,
                        (bar_x, y - 10),
                        (bar_x + bar_width, y + 10),
                        (0, 255, 0),
                        -1
                    )

                    cv2.putText(
                        img,
                        f"{percent}%",
                        (bar_x + bar_width + 5, y),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.5,
                        (255, 255, 255),
                        1
                    )

            except Exception as e:
                print("Error:", e)

    else:

        cv2.putText(
            img,
            "No Pose Detected",
            (20, 50),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 0, 255),
            3
        )

    cv2.imshow("Pose Prediction", img)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

video.release()
cv2.destroyAllWindows()

c:\Users\sreer\OneDrive\Desktop\Luminar\6.DeepLearning\.venv\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step


c:\Users\sreer\OneDrive\Desktop\Luminar\6.DeepLearning\.venv\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step


c:\Users\sreer\OneDrive\Desktop\Luminar\6.DeepLearning\.venv\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


...

c:\Users\sreer\OneDrive\Desktop\Luminar\6.DeepLearning\.venv\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step


c:\Users\sreer\OneDrive\Desktop\Luminar\6.DeepLearning\.venv\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
c:\Users\sreer\OneDrive\Desktop\Luminar\6.DeepLearning\.venv\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step


c:\Users\sreer\OneDrive\Desktop\Luminar\6.DeepLearning\.venv\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
c:\Users\sreer\OneDrive\Desktop\Luminar\6.DeepLearning\.venv\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
